# EDA – Segmentacja i profilowanie regionalnych rynków motoryzacyjnych w Polsce

**Zespół:** Check Engine Team · **Klient:** Sieć autokomisów

Notebook uruchamia się od zera. Ścieżki są względne do katalogu `notebooks/`:
- dane surowe: `../data/raw/*.csv` (paczki roczne 2023–2025, wszystkie 16 województw),
- dane przetworzone (opcjonalnie): `../data/processed/`.

Nazwy kolumn pochodzą z API CEPiK i są przypisane w słowniku `COLS` (sekcja 1) – jeśli w eksporcie nazwy są inne, wystarczy zmienić je tam.

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    display
except NameError:  # poza Jupyterem (np. skrypt)
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Mapowanie nazw kolumn CEPiK -> nazwy robocze
COLS = {
    "kategoria": "kategoria-pojazdu",
    "marka": "marka",
    "rok_produkcji": "rok-produkcji",
    "data_rej": "data-pierwszej-rejestracji-w-kraju",
    "woj_rej": "rejestracja-wojewodztwo",
    "woj_wlasc": "wlasciciel-wojewodztwo",
    "paliwo": "rodzaj-paliwa",
    "paliwo_alt": "rodzaj-paliwa-alternatywnego",
    "pojemnosc": "pojemnosc-silnika",
    "masa": "masa-wlasna",
    "co2": "poziom-emisji-co2",
}
print("Ścieżka danych surowych:", RAW_DIR)

## 2. Wczytanie danych
Łączymy wszystkie paczki roczne w jeden `DataFrame`. Kolumna `plik` zachowuje źródło do późniejszej kontroli jakości.

In [ ]:
files = sorted(glob.glob(str(RAW_DIR / "*.csv")))
assert files, f"Brak plików CSV w {RAW_DIR}. Umieść paczki CEPiK w data/raw/."

def read_one(path):
    # Automatyczne wykrycie separatora (CEPiK eksportuje zwykle ';')
    with open(path, encoding="utf-8", errors="replace") as f:
        sep = ";" if f.readline().count(";") > f.readline().count(",") else ","
    d = pd.read_csv(path, sep=sep, low_memory=False, dtype=str)
    d["plik"] = Path(path).name
    return d

frames = [read_one(p) for p in files]
df_raw = pd.concat(frames, ignore_index=True)
print(f"Wczytano {len(files)} plików: {[Path(f).name for f in files]}")
df_raw.shape

## 3. Podstawowy rzut oka
`shape`, `dtypes` i `describe()` dla danych surowych.

In [ ]:
print("Kształt (wiersze, kolumny):", df_raw.shape)
print()
display(df_raw.dtypes.to_frame("dtype"))

In [ ]:
# Statystyki opisowe – kolumny numeryczne po konwersji
num_cols = [COLS["pojemnosc"], COLS["masa"], COLS["co2"], COLS["rok_produkcji"]]
tmp = df_raw[[c for c in num_cols if c in df_raw.columns]].apply(pd.to_numeric, errors="coerce")
display(tmp.describe().T)

## 4. Brakujące wartości
Liczba i odsetek braków dla każdej kolumny oraz wizualizacja wzorca braków.

In [ ]:
miss = pd.DataFrame({
    "braki_n": df_raw.isna().sum(),
    "braki_%": df_raw.isna().mean() * 100,
}).sort_values("braki_%", ascending=False)
display(miss.round(2))

if HAS_MSNO:
    msno.matrix(df_raw.sample(min(2000, len(df_raw)), random_state=0), figsize=(12, 4), fontsize=8)
    plt.title("Wzorzec brakujących wartości (próbka 2000 wierszy)")
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(df_raw.sample(min(2000, len(df_raw)), random_state=0).isna(),
                cbar=False, ax=ax)
    ax.set_title("Wzorzec brakujących wartości (próbka 2000 wierszy, ciemne = brak)")
    ax.set_xlabel("Kolumna"); ax.set_ylabel("Obserwacja (próbka)")
plt.tight_layout(); plt.show()

**Interpretacja:** _(uzupełnić po uruchomieniu – np. które kolumny są niemal puste, czy braki są skorelowane między kolumnami, np. z rokiem produkcji.)_

## 5. Czyszczenie i inżynieria cech
Zgodnie z raportem: filtr kategorii SAMOCHÓD OSOBOWY, usunięcie masy poza przedziałem 500–3500 kg i pojemności > 7000 cm³, wyliczenie `wiek_pojazdu` i flagi pochodzenia.

In [ ]:
df = df_raw.copy()

# Konwersje typów
df["pojemnosc_cm3"] = pd.to_numeric(df[COLS["pojemnosc"]], errors="coerce")
df["masa_kg"] = pd.to_numeric(df[COLS["masa"]], errors="coerce")
df["co2_g_km"] = pd.to_numeric(df[COLS["co2"]], errors="coerce")
df["rok_produkcji"] = pd.to_numeric(df[COLS["rok_produkcji"]], errors="coerce")
df["rok_rej_pl"] = pd.to_datetime(df[COLS["data_rej"]], errors="coerce").dt.year
df["woj"] = df[COLS["woj_rej"]].str.strip().str.lower()
df["marka"] = df[COLS["marka"]].str.strip().str.upper()
df["paliwo"] = df[COLS["paliwo"]].fillna("BRAK").str.strip().str.upper()

# Filtr: tylko samochody osobowe
mask_cat = df[COLS["kategoria"]].str.upper().str.contains("SAMOCHÓD OSOBOWY", na=False)
df = df[mask_cat].copy()

# Outliery z raportu
mask_ok = (
    df["masa_kg"].between(500, 3500)
    & (df["pojemnosc_cm3"] <= 7000)
    & (df["pojemnosc_cm3"] > 0)
)
print(f"Usunięto {(~mask_ok).sum():,} rekordów odstających (masa / pojemność).")
df = df[mask_ok].copy()

# Cechy docelowe
df["wiek_pojazdu"] = df["rok_rej_pl"] - df["rok_produkcji"]
df = df[df["wiek_pojazdu"].between(0, 60)]
df["pochodzenie"] = np.where(df["wiek_pojazdu"] > 1, "Import używany", "Krajowy (Nowy)")
df["diesel"] = df["paliwo"].str.contains("OLEJ NAPĘDOWY|DIESEL", regex=True, na=False)

print("Po czyszczeniu:", df.shape)
df.to_csv(PROC_DIR / "pojazdy_clean.csv", index=False)

## 6. Podstawowy rzut oka – dane oczyszczone

In [ ]:
print("Kształt:", df.shape)
display(df[["pojemnosc_cm3", "masa_kg", "co2_g_km", "wiek_pojazdu"]].describe().T)

## 7. Rozkłady zmiennych ciągłych
Histogram i boxplot dla wieku, pojemności i masy. Zmienne sięgają różnych skal, więc każda ma własny wykres.

In [ ]:
cont = {
    "wiek_pojazdu": "Wiek pojazdu w chwili rejestracji w PL (lata)",
    "pojemnosc_cm3": "Pojemność silnika (cm³)",
    "masa_kg": "Masa własna pojazdu (kg)",
}
fig, axes = plt.subplots(len(cont), 2, figsize=(12, 3.5 * len(cont)))
for i, (col, label) in enumerate(cont.items()):
    s = df[col].dropna()
    sns.histplot(s, bins=40, kde=True, ax=axes[i, 0], color="steelblue")
    axes[i, 0].set_title(f"Histogram: {col}")
    axes[i, 0].set_xlabel(label); axes[i, 0].set_ylabel("Liczba pojazdów")
    sns.boxplot(x=s, ax=axes[i, 1], color="lightsteelblue", fliersize=2)
    axes[i, 1].set_title(f"Boxplot: {col}")
    axes[i, 1].set_xlabel(label)
plt.tight_layout(); plt.show()

**Interpretacja:** _(uzupełnić – np. czy rozkład wieku jest dwumodalny, co sugerowałoby dwa segmenty: nowe i importowane.)_

## 8. Zmienne kategoryczne
Tabele częstości i wykresy słupkowe: województwo, marka (Top 10), rodzaj paliwa, pochodzenie.

In [ ]:
for col, title in [("woj", "Województwo (rejestracja)"),
                   ("marka", "Marka (Top 10)"),
                   ("paliwo", "Rodzaj paliwa")]:
    freq = df[col].value_counts(dropna=False)
    if col == "marka":
        freq = freq.head(10)
    display(pd.DataFrame({"n": freq, "%": (freq / len(df) * 100).round(2)}))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, n) in zip(axes, [("woj", None), ("marka", 10), ("paliwo", None)]):
    vc = df[col].value_counts()
    if n: vc = vc.head(n)
    sns.barplot(x=vc.values, y=vc.index, ax=ax, color="steelblue")
    ax.set_title(f"Liczba rejestracji wg: {col}")
    ax.set_xlabel("Liczba pojazdów"); ax.set_ylabel(col)
plt.tight_layout(); plt.show()

**Interpretacja:** _(uzupełnić – które województwa i marki dominują; czy paliwo „BRAK” jest istotnym odsetkiem.)_

## 9. Macierz korelacji
Korelacje Pearsona między zmiennymi numerycznymi.

In [ ]:
num = df[["wiek_pojazdu", "pojemnosc_cm3", "masa_kg", "co2_g_km", "rok_produkcji"]]
corr = num.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            square=True, cbar_kws={"label": "Współczynnik Pearsona"}, ax=ax)
ax.set_title("Macierz korelacji zmiennych numerycznych")
plt.tight_layout(); plt.show()

**Interpretacja:** _(uzupełnić – np. silna ujemna korelacja wieku z rokiem produkcji jest oczekiwana z definicji; istotne są korelacje pojemności z masą i wiekiem.)_

## 10. Zależności dwuwymiarowe istotne dla pytań badawczych
Pytania: (a) czy większe silniki występują częściej w starszych autach importowanych, (b) jak odsetek importu i diesla różni się między województwami.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df.sample(min(20000, len(df)), random_state=0),
                x="wiek_pojazdu", y="pojemnosc_cm3", hue="pochodzenie",
                alpha=0.3, s=12, ax=ax)
ax.set_title("Pojemność silnika a wiek pojazdu w chwili rejestracji (próbka 20 000)")
ax.set_xlabel("Wiek pojazdu w chwili rejestracji w PL (lata)")
ax.set_ylabel("Pojemność silnika (cm³)")
ax.legend(title="Pochodzenie")
plt.tight_layout(); plt.show()

**Interpretacja:** _(uzupełnić – czy segment importowany ma większe jednostki napędowe.)_

In [ ]:
reg = (df.groupby("woj")
         .agg(n=("woj", "size"),
              udzial_importu=("pochodzenie", lambda s: (s == "Import używany").mean() * 100),
              udzial_diesla=("diesel", lambda s: s.mean() * 100))
         .query("n >= 500")
         .sort_values("udzial_importu", ascending=False))
display(reg.round(1))

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=reg.reset_index(), x="udzial_importu", y="woj", color="steelblue", ax=ax)
ax.set_title("Odsetek aut z importu wg województwa rejestracji")
ax.set_xlabel("Udział pojazdów importowanych (%)"); ax.set_ylabel("Województwo")
plt.tight_layout(); plt.show()

**Interpretacja:** _(uzupełnić – które regiony są „importowe”; uwaga na województwa mazowieckie i wielkopolskie, gdzie rejestracje poleasingowe zniekształcają obraz.)_

## 11. Podsumowanie
- Główne ubytki danych i sposób ich obsługi (sekcja 4–5).
- Kluczowe obserwacje dla pytania badawczego (sekcje 7–10).
- Wnioski dla etapu modelowania: kandydaci do klasteryzacji K-Means (po wykluczeniu województw mazowieckiego) oraz cechy do klasyfikatora import/salon.

_(uzupełnić po uruchomieniu notebooka)_